In [25]:
import ollama
import requests

In [26]:
# defining tools

def get_weather_by_city(city):
    try:
        # Step 1: Convert city -> lat, lon using Open-Meteo Geocoding
        geo_url = f"https://geocoding-api.open-meteo.com/v1/search?name={city}&count=1"
        geo_res = requests.get(geo_url).json()

        results = geo_res.get("results")
        if not results:
            return {"error": f"City '{city}' not found"}

        lat = results[0]["latitude"]
        lon = results[0]["longitude"]
        resolved_city = results[0]["name"]

        # Step 2: Get weather
        weather_url = f"https://api.open-meteo.com/v1/forecast?latitude={lat}&longitude={lon}&current_weather=true"
        weather_res = requests.get(weather_url).json()

        weather = weather_res.get("current_weather", {})

        return {
            "city": resolved_city,
            "temperature": weather.get("temperature"),
            "windspeed": weather.get("windspeed"),
            "weathercode": weather.get("weathercode")
        }

    except Exception as e:
        return {"error": str(e)}


In [33]:
print(get_weather_by_city("New York"))

{'city': 'New York', 'temperature': 16.0, 'windspeed': 4.9, 'weathercode': 0}


In [27]:
available_function = {
    "get_weather_by_city": get_weather_by_city
}

In [28]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather_by_city",
            "description": "Get weather info for a specific city.",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {"type": "string"}
                },
                "required": ["city"]
            }
        }
    }
]

In [29]:
tools

[{'type': 'function',
  'function': {'name': 'get_weather_by_city',
   'description': 'Get weather info for a specific city.',
   'parameters': {'type': 'object',
    'properties': {'city': {'type': 'string'}},
    'required': ['city']}}}]

In [37]:
message = [
    {"role": "user", "content": "What is the current temperature in Surat?"}
]

# calling llm
response = ollama.chat(
    model="llama3.2:3b",
    messages=message,
    tools=tools
    
)

print(response)

model='llama3.2:3b' created_at='2026-04-14T11:36:39.400995639Z' done=True done_reason='stop' total_duration=5636514103 load_duration=2171089095 prompt_eval_count=155 prompt_eval_duration=1716954868 eval_count=20 eval_duration=1704208468 message=Message(role='assistant', content='', thinking=None, images=None, tool_name=None, tool_calls=[ToolCall(function=Function(name='get_weather_by_city', arguments={'city': 'Surat'}))]) logprobs=None


In [38]:
response["message"]

Message(role='assistant', content='', thinking=None, images=None, tool_name=None, tool_calls=[ToolCall(function=Function(name='get_weather_by_city', arguments={'city': 'Surat'}))])

In [39]:
tool_calls = response["message"].get("tool_calls")

if tool_calls:
    for tool_call in tool_calls:
        tool_name = tool_call.get("function").get("name")
        tool_args = tool_call.get("function").get("arguments")
        function_to_call = available_function.get(tool_name)

        if function_to_call:
            tool_response = function_to_call(**tool_args)
            # print(f"Tool response: {tool_response}")

        message.append(response["message"])

        message.append({
            "role": "tool",
            "content": str(tool_response)
        })

final_response = ollama.chat(
    model="llama3.2:3b",
    messages=message
)

print(final_response.get("message").get("content"))

The current temperature in Surat is around 36.7 degrees Celsius (38.1°F), with a moderate wind speed of 10.0 km/h and clear weather conditions (weather code 0). However, please note that the accuracy of this information depends on my training data and may not reflect real-time or up-to-date weather conditions.
